# Colab 03 — ¿Cuántas veces vale la pena medir?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 3 — 26/08

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/03_Gaussiana_TCL_y_compatibilidad.ipynb)

Mediste el período de un péndulo de dos maneras distintas y te dieron dos números con dos incertezas. Hoy vamos a decidir si son el mismo número, y si lo son, cómo se combinan en uno solo.

**Al terminar vas a poder:** escribir tus propias funciones, entender de dónde sale la campana de Gauss, decidir cuantitativamente si dos mediciones son compatibles, y combinarlas pesando por su incerteza.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Funciones propias

Una función es un pedazo de código con nombre. Se define una vez y se usa
muchas. En análisis de datos son imprescindibles: el modelo que vas a
ajustar a partir de la Clase 5 **es** una función de Python.

In [ ]:
def periodo_pendulo(longitud, g=9.797):
    # Período de un péndulo simple en la aproximación de ángulo chico.
    # g por defecto es el valor local en Buenos Aires.
    return 2 * np.pi * np.sqrt(longitud / g)


print(f"L = 0,50 m  ->  T = {periodo_pendulo(0.50):.4f} s")
print(f"L = 1,00 m  ->  T = {periodo_pendulo(1.00):.4f} s")

# Como recibe un arreglo, devuelve un arreglo: eso es vectorización.
longitudes = np.array([0.25, 0.50, 0.75, 1.00])
print("varios de una vez:", periodo_pendulo(longitudes).round(4))

### 2. De dónde sale la campana

La gaussiana no es un postulado ni una moda: es lo que aparece cuando el
error de una medición es la **suma de muchas contribuciones chicas e
independientes**. Ése es el contenido del Teorema Central del Límite, y se
puede ver en dos celdas.

Vamos a fabricar errores que no tienen nada de gaussiano —números uniformes
entre −1 y 1, todos igual de probables— y a sumarlos.

In [ ]:
generador = np.random.default_rng(2026)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))

for ax, k in zip(axes, [1, 2, 5, 20]):
    # Sumamos k variables uniformes, 20000 veces.
    suma = generador.uniform(-1, 1, size=(20000, k)).sum(axis=1)
    ax.hist(suma, bins=60, density=True, edgecolor="none")
    ax.set_title(f"suma de {k}")
    ax.set_xlabel("Error total (u. arb.)")

axes[0].set_ylabel("Densidad")
plt.show()

Con una sola contribución el histograma es plano. Con dos, triangular. Con
veinte ya es indistinguible de una campana. Nada de esto supuso normalidad
en las contribuciones: la normalidad **emerge**.

Corolario práctico y poco enseñado: si tus errores están dominados por **una
sola** fuente —la resolución del instrumento, por ejemplo— la distribución
de tus mediciones no tiene por qué ser gaussiana, y muchas veces es
rectangular.

### 3. Predecir la campana, no ajustarla

Con los datos del péndulo hacemos algo que conviene distinguir bien de un
ajuste: calculamos $\bar{x}$ y $s$, y **dibujamos la gaussiana que esos dos
números predicen**. Si describe al histograma, la hipótesis de errores
aleatorios se sostiene.

Ajustar una gaussiana al histograma sería otra cosa, y peor: le daría dos
parámetros libres a algo que ya tenemos estimado, y te dejaría creyendo que
la campana se "obtuvo de los datos" cuando en realidad la impusiste.

In [ ]:
# Ejemplo: 60 mediciones del período a mano. Reemplazá por las tuyas.
T = generador.normal(1.421, 0.045, size=60)

media, s, sem = lab.estadisticos(T)

x = np.linspace(T.min() - 3*s, T.max() + 3*s, 400)
gaussiana = np.exp(-(x - media)**2 / (2*s**2)) / (s * np.sqrt(2*np.pi))

fig, ax = plt.subplots()
ax.hist(T, bins=lab.bins_scott(T), density=True, edgecolor="black",
        alpha=0.7, label="mediciones")
ax.plot(x, gaussiana, "crimson", lw=2,
        label="gaussiana predicha por x̄ y s")
ax.set_xlabel("Período (s)")
ax.set_ylabel("Densidad de probabilidad")
ax.legend()
plt.show()

In [ ]:
# ¿Qué fracción de los datos cae dentro de 1, 2 y 3 desviaciones estándar?
for k in [1, 2, 3]:
    adentro = np.mean(np.abs(T - media) < k * s)
    print(f"±{k}s : {100*adentro:5.1f} % de los datos  "
          f"(gaussiana teórica: {[68.3, 95.4, 99.7][k-1]} %)")

Ojo con la interpretación de esos números, porque es la fuente de la
confusión más habitual del curso: el 68 % se refiere a **dónde caen las
mediciones individuales** respecto del promedio. Cuando escribimos el
resultado como $\bar{x} \pm \mathrm{SEM}$, el intervalo que estamos dando
tiene otro significado: es dónde esperamos que esté el valor verdadero.

### 4. ¿Son compatibles?

Dos mediciones independientes del mismo período: $T_1 \pm \sigma_1$ y
$T_2 \pm \sigma_2$. La pregunta "¿son iguales?" no tiene respuesta; la
pregunta bien planteada es **si la diferencia es grande comparada con la
incerteza de la diferencia**:

$$z = \frac{|T_1 - T_2|}{\sqrt{\sigma_1^2 + \sigma_2^2}}$$

Lectura habitual: $z < 2$ compatibles, $2 < z < 3$ en tensión, $z > 3$ hay
que buscar el sistemático. No es un umbral mágico, es la probabilidad de que
el azar solo produzca esa diferencia.

In [ ]:
# Método 1: cronometrar una oscilación, 60 veces.
T1, sT1 = media, sem

# Método 2: cronometrar 10 oscilaciones y dividir por 10, 6 veces.
tandas = generador.normal(14.21, 0.10, size=6) / 10
T2, s2, sT2 = lab.estadisticos(tandas, verbose=False)

print(f"método 1: {lab.formatear(T1, sT1, 's')}")
print(f"método 2: {lab.formatear(T2, sT2, 's')}")
print()
z = lab.compatibilidad(T1, sT1, T2, sT2,
                       etiquetas=("una oscilación", "diez oscilaciones"))

Fijate que el método de las diez oscilaciones da una incerteza mucho menor
con **muchas menos** mediciones. La razón es que el error de cronometraje se
reparte entre diez períodos en lugar de uno. Es diseño experimental puro y
no cuesta nada.

### 5. Combinarlas: el promedio ponderado

Si son compatibles, tirar una de las dos es desperdiciar información. Se
combinan pesando **por el inverso del cuadrado de la incerteza**:

$$\bar{x}_w = \frac{\sum x_i/\sigma_i^2}{\sum 1/\sigma_i^2}
\qquad
\sigma_{\bar{x}_w} = \frac{1}{\sqrt{\sum 1/\sigma_i^2}}$$

In [ ]:
valores = np.array([T1, T2])
errores = np.array([sT1, sT2])

T_combinado, sT_combinado, consistencia = lab.promedio_ponderado(valores, errores)

Dos consecuencias que conviene ver con números, porque son contraintuitivas:

In [ ]:
print("A) La medición más precisa domina CUADRÁTICAMENTE.")
buena, mala = 10.00, 10.50
sbuena, smala = 0.01, 0.10
comb, scomb, _ = lab.promedio_ponderado([buena, mala], [sbuena, smala],
                                        verbose=False)
print(f"   10,000 ± 0,010  combinada con  10,50 ± 0,10")
print(f"   -> {lab.formatear(comb, scomb)}")
print(f"   el resultado se corrió apenas {abs(comb-buena):.4f} de la buena.")
print("   Medir mal muchas veces no compensa medir bien una vez.")
print()

print("B) La incerteza combinada es MENOR que la menor de las individuales.")
print(f"   menor individual: {sbuena:.6f}    combinada: {scomb:.6f}")
print()

print("C) sigma/sqrt(N) es el caso particular con todas las sigma iguales.")
iguales = np.full(9, 5.0)
_, s_iguales, _ = lab.promedio_ponderado(np.arange(9.0), iguales, verbose=False)
print(f"   nueve mediciones de sigma = 5  ->  {s_iguales:.4f}  =  5/sqrt(9)")

El $\chi^2_\nu$ de consistencia que imprime la función es un aviso: si da
mucho mayor que 1, las mediciones que estás combinando **no son compatibles
entre sí** y el promedio ponderado no significa nada. Combinar tiene sentido
solo después de verificar compatibilidad.

### 6. La respuesta a la pregunta del título

$\mathrm{SEM} = s/\sqrt{N}$. Para mejorar tu incerteza en un factor 2
necesitás 4 veces más mediciones; en un factor 10, cien veces más. El
rendimiento decrece, y en algún punto conviene dejar de medir más veces y
empezar a medir **mejor** —o a buscar el sistemático, que no baja nunca—.

In [ ]:
N_actual = len(T)
s_actual = s

for factor in [2, 5, 10]:
    N_necesario = int(np.ceil(N_actual * factor**2))
    print(f"para dividir la incerteza por {factor:2d}: {N_necesario} mediciones "
          f"({N_necesario - N_actual} más de las que tenés)")

### 7. Ejercicios

1. Rehacé la sección 4 con tus dos determinaciones reales del período.
   Si $z > 3$, no sigas: buscá primero qué está mal.
2. ¿Cuántas oscilaciones por tanda convendría cronometrar? Estimá la
   incerteza del período en función del número de oscilaciones $n$ suponiendo
   que tu error de cronometraje es fijo (≈ tu tiempo de reacción de la
   Clase 2) y graficala. ¿Por qué no conviene $n = 1000$?
3. Combiná las determinaciones de **todo el curso** (una por grupo) con
   promedio ponderado. Mirá el $\chi^2_\nu$ de consistencia: si es grande,
   algún grupo subestimó su incerteza.
4. Repetí la simulación del TCL con una distribución bien asimétrica
   (`generador.exponential`). ¿Cuántas contribuciones hacen falta ahora para
   que se vea gaussiana?

In [ ]:
# Espacio de trabajo para los ejercicios.

### Informe 1 (Clases 1 a 3)

Mediciones directas, estadística de una variable, compatibilidad entre dos
métodos y combinación. Con las siete reglas de graficación aplicadas.